In [15]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
import numpy as np
import cv2

In [16]:
train_gen = ImageDataGenerator(rescale=1./255, validation_split = 0.25)
test_gen = ImageDataGenerator(rescale=1./255)

In [41]:
train_data = train_gen.flow_from_directory(
    "face_detection_dataset/train_data",
    target_size = (128,128),
    batch_size=32,
    class_mode='categorical',
    subset = 'training'

)

test_data = train_gen.flow_from_directory(
    "face_detection_dataset/test_data",
    target_size = (128,128),
    batch_size=32,
    class_mode='categorical',
    subset = 'validation'    

)   

Found 1741 images belonging to 1 classes.
Found 90 images belonging to 1 classes.


In [42]:
# Check which class is 0 and which is 1
print(train_data.class_indices)

{'train_data': 0}


## Building the CNN Model

In [43]:
model = Sequential([
    # First Convolutional Block
    Conv2D(32, (3,3), activation='relu', input_shape=(128, 128, 3)),
    MaxPooling2D(2,2),
    
    # Second Convolutional Block
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    
    # Third Convolutional Block (Helps catch deeper features)
    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    
    # Flattening and Dense Layers
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),  # Helps prevent overfitting
    
    # Output Layer
    # We use 2 neurons because you have 2 classes ('it_sme', 'it_snotme')
    Dense(2, activation='softmax') 
])

# Compile the model
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Summary to check the architecture
model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d_12 (Conv2D)                   │ (None, 126, 126, 32)        │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_12 (MaxPooling2D)      │ (None, 63, 63, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_13 (Conv2D)                   │ (None, 61, 61, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_13 (MaxPooling2D)      │ (None, 30, 30, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_14 (Conv2D)                   │ (None, 28, 28, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_14 (MaxPooling2D)      │ (None, 14, 14, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_4 (Flatten)                  │ (None, 25088)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_8 (Dense)                      │ (None, 128)                 │       3,211,392 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_4 (Dropout)                  │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_9 (Dense)                      │ (None, 2)                   │             258 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 3,304,898 (12.61 MB)

 Trainable params: 3,304,898 (12.61 MB)

 Non-trainable params: 0 (0.00 B)

In [44]:
# Train the model
history = model.fit(
    train_data,
    validation_data=test_data,
    epochs=15  
)

Epoch 1/15


ValueError: Arguments `target` and `output` must have the same shape. Received: target.shape=(None, 1), output.shape=(None, 2)

In [ ]:
model.save('my_face_model.keras')
print("Saved as .keras")

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image

# 1. Load an image to test
img_path = "train_data/train_data/it_sme/1 (1).jpg" 

# 2. Preprocess the image (Must match training: 128x128, normalized)
img = image.load_img(img_path, target_size=(128, 128))
img_array = image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0) # Add batch dimension
img_array = img_array / 255.0  # Normalize pixel values

# 3. Make Prediction
predictions = loaded_model.predict(img_array)
score = predictions[0]

# 4. Interpret Result (Based on your earlier class indices)
# {'it_sme': 0, 'it_snotme': 1}
if score[0] > score[1]:
    print(f"Result: It's ME! (Confidence: {score[0]*100:.2f}%)")
else:
    print(f"Result: It's NOT ME. (Confidence: {score[1]*100:.2f}%)")

In [ ]:
# import tensorflow as tf
# from tensorflow.keras.preprocessing.image import ImageDataGenerator
# from tensorflow.keras.models import Sequential, load_model
# from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
# import numpy as np
# import cv2

In [ ]:
# train_gen = ImageDataGenerator(rescale=1./255, validation_split = 0.25)

In [ ]:
# train_data = train_gen.flow_from_directory(
#     "train_data/train_data",
#     target_size = (128,128),
#     batch_size=32,
#     class_mode='categorical',
#     subset = 'training'

# )

# validation = train_gen.flow_from_directory(
#     "train_data/train_data",
#     target_size = (128,128),
#     batch_size=32,
#     class_mode='categorical',
#     subset = 'validation'    

# )  

In [ ]:
# model = Sequential()
# model.add(Conv2D(124,(3,3), activation='relu',input_shape=(128,128,3)))
# model.add(MaxPooling2D(2,2))
# model.add(Conv2D(64,(3,3), activation='relu')
# model.add(MaxPooling2D(2,2))
# model.add(Conv2D(64,(3,3), activation='relu')
# model.add(MaxPooling2D(2,2))          
# model.add(Conv2D(32,(3,3), activation='relu')
# model.add(MaxPooling2D(2,2))          

In [ ]:
# model.add(Flatten())
# model.add(Dense(64,activation='relu'))
# model.add(Dense(64,activation='relu'))
# model.add(Dense(2,activation='softmax'))

In [ ]:
# model.summary()

In [ ]:
# model.compile(optimizer = 'adam',
#              loss= 'categorical-crossentropy',
#              metrics= ['accuracy'],
               )

In [ ]:
# model.fit(train_data, epochs=10,verbose=1,validation_data=validation=data)

In [ ]:
# model.save('models/facerecognition.keras')


In [ ]:
# model = load_model('models/facerecognition.keras')

In [ ]:
# label_map = train_data.class_indices:
# inv_label_map {v:k for k, v in label_map.items()}
# print("Label mapping:", inv_label_map)

In [ ]:
# def predict image(path):
#     img = cv2.imread(path)
#     img = cv2.resize(img, (128, 128))
#     img = img / 255.0
#     img = np.expand_dims(img, axis = 0)

#     preds = model.predict(img)
#     Index = np.argmax(preds)         #no if/eise
#     label = inv_label_map[index]     #label from samping


#     print("\nPrediction Scores:", preds)
#     print("Predicted Label:", label)

In [ ]:
# predict image('1.jpg')